In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

DATA_DIR = Path("../training_datasets")
OUTPUT_DIR = Path("../cleaned_datasets")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

# Business Name Normalization

In [2]:
HONORIFICS = set()  # mr/mrs/ms/dr all removed — too many real businesses are branded
                     # around a title word (Dr. Reddy's Laboratories, Mrs. Fields Cookies)
                     # for a blind strip to be safe under a precision-weighted metric.

COMPOUND_SUFFIXES = {
    ('private', 'limited'),
    ('pvt', 'ltd'),
    ('pvt', 'limited'),
}

SUFFIXES = {
    'inc', 'incorporated', 'ltd', 'limited', 'llc', 'corp', 'corporation',
    'co', 'company', 'pvt', 'llp', 'plc'
}

DOTTED_SUFFIX_MAP = {
    r'\bl\.?\s*l\.?\s*c\.?\b': 'llc',
    r'\bp\.?\s*l\.?\s*c\.?\b': 'plc',
    r'\bl\.?\s*t\.?\s*d\.?\b': 'ltd',
    r'\binc\.?\b': 'inc',
    r'\bcorp\.?\b': 'corp',
    r'\bco\.?\b': 'co',
}

def normalize_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ''

    s = unicodedata.normalize('NFKC', name)
    s = s.lower()

    for pattern, replacement in DOTTED_SUFFIX_MAP.items():
        s = re.sub(pattern, replacement, s)

    ampersand_replaced = '&' in s
    s = s.replace('&', ' and ')

    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()

    tokens = s.split()

    while tokens and tokens[0] in HONORIFICS:
        tokens.pop(0)

    for compound in COMPOUND_SUFFIXES:
        n = len(compound)
        if len(tokens) >= n and tuple(tokens[-n:]) == compound:
            tokens = tokens[:-n]
            break

    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()

    if ampersand_replaced and tokens and tokens[-1] == 'and':
        tokens.pop()

    return ' '.join(tokens)

In [3]:
test_names = [
    "Dr. Sharma & Co.",
    "Sharma & Company",
    "SRI RAM PRIVATE LIMITED",
    "SRI RAM PVT LTD",
    "Tata Motors Ltd.",
    "Apple Inc.",
    "Microsoft Corporation",
    "ABC L.L.C.",
    "ABC L T D",
    "ABC P.L.C.",
    "MÜLLER & SÖHNE",
    "Dr.   Sharma   &   Co.",
    "Sharma and Sons",
    "Sri Lakshmi Traders",
]

for name in test_names:
    print(f"{name!r:40s} → {normalize_name(name)!r}")

'Dr. Sharma & Co.'                       → 'dr sharma'
'Sharma & Company'                       → 'sharma'
'SRI RAM PRIVATE LIMITED'                → 'sri ram'
'SRI RAM PVT LTD'                        → 'sri ram'
'Tata Motors Ltd.'                       → 'tata motors'
'Apple Inc.'                             → 'apple'
'Microsoft Corporation'                  → 'microsoft'
'ABC L.L.C.'                             → 'abc'
'ABC L T D'                              → 'abc'
'ABC P.L.C.'                             → 'abc'
'MÜLLER & SÖHNE'                         → 'müller and söhne'
'Dr.   Sharma   &   Co.'                 → 'dr sharma'
'Sharma and Sons'                        → 'sharma and sons'
'Sri Lakshmi Traders'                    → 'sri lakshmi traders'


# Business Address Normalization

In [4]:
def normalize_address(addr: str) -> str:
    if not isinstance(addr, str) or not addr.strip():
        return ''
    s = unicodedata.normalize('NFKC', addr)
    s = s.lower()
    s = re.sub(r'[^\w\s]', ' ', s)       # punctuation → space
    s = re.sub(r'\s+', ' ', s).strip()   # collapse whitespace
    return s


def extract_zip_pin(addr: str, country: str) -> str:
    """Extract from the *raw* address, not the cleaned one."""
    if not isinstance(addr, str):
        return ''
    if country == 'US':
        # anchored to the end — a ZIP comes after city/state, never at the front
        m = re.search(r'\b(\d{5})(?:-\d{4})?\s*$', addr)
    elif country == 'India':
        m = re.search(r'\b(\d{6})\b', addr)   # unchanged — not reported as a problem
    else:
        return ''
    return m.group(1) if m else ''


def extract_leading_number(addr: str, country: str = '') -> str:
    if not isinstance(addr, str):
        return ''

    m = re.match(r'^\W*(\d+[A-Za-z]?)', addr)   # \W* instead of \s* — also skips junk
    if not m:                                    # like the '##' prefix your own tests hit
        return ''

    num = m.group(1)

    # India: a PIN can legitimately be the first token ("560034, Bangalore, Karnataka"),
    # and a 6-digit plot/door number is essentially never real — still excluded.
    if country == 'India' and re.fullmatch(r'\d{6}', num):
        return ''

    # US: no exclusion anymore. extract_zip_pin only matches at the *end* now, so a
    # leading 5-digit number can't be claimed by both functions at once — nothing left
    # to guard against.
    return num

In [5]:
test_cases = [
    # (raw_address, country, expected_clean, expected_zip_pin, expected_leading)
    ("123 Main Street, New York, NY 10001",          "US",    "123 main street new york ny 10001", "10001", "123"),
    ("123 Main Street, New York, NY 10001-1234",     "US",    "123 main street new york ny 10001 1234", "10001", "123"),
    ("560034, Bangalore, Karnataka",                 "India", "560034 bangalore karnataka", "560034", ""),
    ("45A MG Road, Bengaluru 560001",                "India", "45a mg road bengaluru 560001", "560001", "45A"),
    ("Plot No. 12, Sector 5",                        "India", "plot no 12 sector 5", "", ""),
    ("Empire State Building, New York",              "US",    "empire state building new york", "", ""),
    ("##120 WOOD THRUSH LN, MOORESVILLE, NC",        "US",    "120 wood thrush ln mooresville nc", "", ""),
    ("No.2A, Saravana Signature Suites, Coimbatore", "India", "no 2a saravana signature suites coimbatore", "", ""),
    ("10001 Broadway, New York",                     "US",    "10001 broadway new york", "10001", ""),
    ("12B, 1st Floor, Andheri East, Mumbai 400069",  "India", "12b 1st floor andheri east mumbai 400069", "400069", "12B"),
    ("",                                             "US",    "", "", ""),
    (None,                                           "India", "", "", ""),
]

print(f"{'Raw':<55} | {'Clean':<45} | {'ZIP/PIN':<8} | {'Lead'}")
print("-" * 120)

for raw, country, exp_clean, exp_zip, exp_lead in test_cases:
    clean = normalize_address(raw)
    z = extract_zip_pin(raw or '', country)
    lead = extract_leading_number(raw or '', country)

    # simple visual check
    ok = (clean == exp_clean) and (z == exp_zip) and (lead == exp_lead)
    status = "✓" if ok else "✗"

    print(f"{status} {str(raw)[:53]:<53} | {clean[:43]:<43} | {z:<8} | {lead}")

Raw                                                     | Clean                                         | ZIP/PIN  | Lead
------------------------------------------------------------------------------------------------------------------------
✓ 123 Main Street, New York, NY 10001                   | 123 main street new york ny 10001           | 10001    | 123
✓ 123 Main Street, New York, NY 10001-1234              | 123 main street new york ny 10001 1234      | 10001    | 123
✓ 560034, Bangalore, Karnataka                          | 560034 bangalore karnataka                  | 560034   | 
✓ 45A MG Road, Bengaluru 560001                         | 45a mg road bengaluru 560001                | 560001   | 45A
✓ Plot No. 12, Sector 5                                 | plot no 12 sector 5                         |          | 
✓ Empire State Building, New York                       | empire state building new york              |          | 
✗ ##120 WOOD THRUSH LN, MOORESVILLE, NC             

In [6]:
def has_non_latin_script(name: str) -> bool:
    if not isinstance(name, str):
        return False
    for char in name:
        if not char.isalpha():
            continue
        char_name = unicodedata.name(char, '')
        if not char_name.startswith('LATIN'):
            return True
    return False

# Final Cleaning Processing Layer

In [7]:
def clean_dataset(input_path, output_path):
    print(f"\nProcessing: {input_path}")
    print(f"Output:    {output_path}")

    # Remove only the previously generated cleaned file.
    # The original dataset is never modified.
    if output_path.exists():
        output_path.unlink()

    first_chunk = True
    total_rows = 0

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False
    ):
        # --------------------------------------------------
        # 1. Clean business name
        # --------------------------------------------------
        chunk["clean_business_name"] = (
            chunk["business_name"].apply(normalize_name)
        )

        # --------------------------------------------------
        # 2. Detect non-Latin script in ORIGINAL name
        # --------------------------------------------------
        chunk["has_non_latin_script"] = (
            chunk["business_name"].apply(has_non_latin_script)
        )

        # --------------------------------------------------
        # 3. Clean business address
        # --------------------------------------------------
        chunk["clean_business_address"] = (
            chunk["business_address"].apply(normalize_address)
        )

        # --------------------------------------------------
        # 4. Extract ZIP/PIN from RAW address
        # --------------------------------------------------
        chunk["zip_pin"] = chunk.apply(
            lambda row: extract_zip_pin(
                row["business_address"],
                row["country"]
            ),
            axis=1
        )

        # --------------------------------------------------
        # 5. Extract leading street number from RAW address
        # --------------------------------------------------
        chunk["leading_number"] = chunk.apply(
            lambda row: extract_leading_number(
                row["business_address"],
                row["country"]
            ),
            axis=1
        )

        # --------------------------------------------------
        # 6. Keep raw + cleaned/derived columns
        # --------------------------------------------------
        output_chunk = chunk[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "clean_business_name",
                "clean_business_address",
                "zip_pin",
                "leading_number",
                "has_non_latin_script"
            ]
        ]

        # --------------------------------------------------
        # 7. Write chunk to output TSV
        # --------------------------------------------------
        output_chunk.to_csv(
            output_path,
            sep="\t",
            index=False,
            mode="w" if first_chunk else "a",
            header=first_chunk
        )

        total_rows += len(chunk)

        print(f"Processed: {total_rows:,} rows")

        first_chunk = False

    print(f"Finished: {total_rows:,} rows")

In [8]:
test_chunk = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    nrows=100_000,
    keep_default_na=False
)

test_chunk["name_clean"] = test_chunk["business_name"].apply(normalize_name)
test_chunk["has_non_latin_script"] = (
    test_chunk["business_name"].apply(has_non_latin_script)
)
test_chunk["address_clean"] = (
    test_chunk["business_address"].apply(normalize_address)
)

test_chunk["zip_pin"] = test_chunk.apply(
    lambda row: extract_zip_pin(
        row["business_address"],
        row["country"]
    ),
    axis=1
)

test_chunk["leading_number"] = test_chunk.apply(
    lambda row: extract_leading_number(
        row["business_address"],
        row["country"]
    ),
    axis=1
)

test_chunk.head(10)

,entity_id,business_name,business_address,country,name_clean,has_non_latin_script,address_clean,zip_pin,leading_number
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,False,1795 westchester drive high point nc,,1795
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,False,17560 ellis road tahlequah ok,,17560
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail,False,1712 montebello avenue phoenix az,,1712
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,False,2100 cameron drive unit apartment g dundalk md,,2100
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,False,797 lake town block a kolkata howrah west bengal,,797
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,False,oh columbus 5559 orville avenue,,
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,False,2505 tower 1 oakwood runwal greens mulund gore...,,2505
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,False,1111 church street unit 2007 nashville tn,,1111
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,False,337 oakland avenue michigan city in,,337
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,False,294 meadowcreek drive unit unit 2 village of p...,,294


In [9]:
clean_dataset(
    DATA_DIR / "train_source1.tsv",
    OUTPUT_DIR / "cleaned_source1.tsv"
)


Processing: ../training_datasets/train_source1.tsv
Output:    ../cleaned_datasets/cleaned_source1.tsv
Processed: 100,000 rows
Processed: 200,000 rows
Processed: 300,000 rows
Processed: 400,000 rows
Processed: 500,000 rows
Processed: 600,000 rows
Processed: 700,000 rows
Processed: 800,000 rows
Processed: 900,000 rows
Processed: 1,000,000 rows
Processed: 1,100,000 rows
Processed: 1,200,000 rows
Processed: 1,300,000 rows
Processed: 1,400,000 rows
Processed: 1,500,000 rows
Processed: 1,600,000 rows
Processed: 1,700,000 rows
Processed: 1,800,000 rows
Processed: 1,900,000 rows
Processed: 2,000,000 rows
Processed: 2,100,000 rows
Processed: 2,200,000 rows
Processed: 2,206,821 rows
Finished: 2,206,821 rows


In [10]:
clean_dataset(
    DATA_DIR / "train_source2.tsv",
    OUTPUT_DIR / "cleaned_source2.tsv"
)


Processing: ../training_datasets/train_source2.tsv
Output:    ../cleaned_datasets/cleaned_source2.tsv
Processed: 100,000 rows
Processed: 200,000 rows
Processed: 300,000 rows
Processed: 400,000 rows
Processed: 500,000 rows
Processed: 600,000 rows
Processed: 700,000 rows
Processed: 800,000 rows
Processed: 900,000 rows
Processed: 1,000,000 rows
Processed: 1,100,000 rows
Processed: 1,200,000 rows
Processed: 1,300,000 rows
Processed: 1,400,000 rows
Processed: 1,500,000 rows
Processed: 1,600,000 rows
Processed: 1,700,000 rows
Processed: 1,800,000 rows
Processed: 1,900,000 rows
Processed: 2,000,000 rows
Processed: 2,100,000 rows
Processed: 2,200,000 rows
Processed: 2,300,000 rows
Processed: 2,400,000 rows
Processed: 2,500,000 rows
Processed: 2,600,000 rows
Processed: 2,700,000 rows
Processed: 2,800,000 rows
Processed: 2,900,000 rows
Processed: 3,000,000 rows
Processed: 3,100,000 rows
Processed: 3,200,000 rows
Processed: 3,300,000 rows
Processed: 3,400,000 rows
Processed: 3,500,000 rows
Proce

In [11]:
clean_dataset(
    DATA_DIR / "train_source3.tsv",
    OUTPUT_DIR / "cleaned_source3.tsv"
)


Processing: ../training_datasets/train_source3.tsv
Output:    ../cleaned_datasets/cleaned_source3.tsv
Processed: 100,000 rows
Processed: 200,000 rows
Processed: 300,000 rows
Processed: 400,000 rows
Processed: 500,000 rows
Processed: 600,000 rows
Processed: 700,000 rows
Processed: 800,000 rows
Processed: 900,000 rows
Processed: 1,000,000 rows
Processed: 1,100,000 rows
Processed: 1,200,000 rows
Processed: 1,300,000 rows
Processed: 1,400,000 rows
Processed: 1,500,000 rows
Processed: 1,600,000 rows
Processed: 1,700,000 rows
Processed: 1,800,000 rows
Processed: 1,900,000 rows
Processed: 2,000,000 rows
Processed: 2,100,000 rows
Processed: 2,200,000 rows
Processed: 2,300,000 rows
Processed: 2,400,000 rows
Processed: 2,500,000 rows
Processed: 2,600,000 rows
Processed: 2,700,000 rows
Processed: 2,800,000 rows
Processed: 2,900,000 rows
Processed: 3,000,000 rows
Processed: 3,100,000 rows
Processed: 3,200,000 rows
Processed: 3,300,000 rows
Processed: 3,400,000 rows
Processed: 3,500,000 rows
Proce